# NETRA Phase 2 — DistilBERT Tier-1 Training

**Project:** NETRA — The AI Eye Against Phishing  
**Phase 2 Goal:** Replace Random Forest with fine-tuned DistilBERT for phishing detection.  
**Expected Results:** Phishing Recall 94–97% | FPR < 3%

---

> **Before running:** `Runtime → Change runtime type → T4 GPU`  
> Estimated training time: 20–25 minutes on Colab free T4.

## Section 1: Environment Setup

Install all required libraries. Run once. Restart runtime if prompted, then skip this cell.

In [ ]:
!pip install transformers==4.40.0 torch scikit-learn pandas numpy imbalanced-learn matplotlib seaborn joblib tqdm --quiet
print("All dependencies installed.")

In [ ]:
from google.colab import drive
from pathlib import Path
import sys, json

drive.mount("/content/drive")

DRIVE_ROOT    = Path("/content/drive/MyDrive/NETRA")
DATA_CSV      = DRIVE_ROOT / "data" / "processed" / "unified.csv"
MODELS_DIR    = DRIVE_ROOT / "ml" / "models"
EVAL_DIR      = DRIVE_ROOT / "ml" / "evaluation"
NOTEBOOKS_DIR = DRIVE_ROOT / "notebooks"
for d in [MODELS_DIR, EVAL_DIR]: d.mkdir(parents=True, exist_ok=True)

if str(DRIVE_ROOT) not in sys.path:
    sys.path.insert(0, str(DRIVE_ROOT))

print("DRIVE_ROOT:", DRIVE_ROOT)
print("Data CSV exists:", DATA_CSV.exists())

## Section 2: Why DistilBERT?

### The Problem with TF-IDF + Random Forest

Phase 1 used 5,000 TF-IDF bag-of-words features + Random Forest. It fails on:
- Short phishing emails with few distinctive keywords
- Typosquatted domains like `paypa1` (unknown token to TF-IDF)
- Social engineering that uses normal words in a manipulative sequence

### How DistilBERT Fixes This

- **Pretrained on 3.3 billion words** — understands English grammar, intent, context
- **Reads whole sentences** — `"verify your account immediately"` is flagged as a unit
- **40% smaller than BERT, 60% faster** — production-ready

### Architecture
```
[Email Body Text]
       |
DistilBERT (6 transformer layers)
       |
[CLS] embedding (768 dims)
       |           [SPF/DKIM/DMARC Features (10)] -> Linear(10->32) -> ReLU
Concatenate [768 + 32 = 800 dims]
       |
Linear(800->256) -> ReLU -> Dropout(0.3) -> Linear(256->2)
       |
[LEGITIMATE=0 / PHISHING=1]
```

**SUSPICIOUS** is inference-time only: if `max(softmax_prob) < 0.70` → output `SUSPICIOUS`

## Section 3: Data Loading & Class Distribution

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

HEADER_FEATURE_NAMES = [
    "spf_pass", "spf_fail", "spf_none",
    "dkim_pass", "dkim_fail", "dkim_none",
    "dmarc_pass", "dmarc_fail", "dmarc_none",
    "sender_domain_match",
]

df = pd.read_csv(DATA_CSV, dtype=str)
df = df[df["split"].isin(["train", "val"])].reset_index(drop=True)
df["label"] = df["label"].astype(int)
for col in HEADER_FEATURE_NAMES:
    if col not in df.columns:
        df[col] = 0
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(float)

print(f'Total: {len(df):,} | Splits: {df["split"].value_counts().to_dict()}')
print(f'Labels: {df["label"].value_counts().to_dict()}')

counts = df["label"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(["LEGITIMATE", "PHISHING"], counts.values, color=["#2ecc71", "#e74c3c"], edgecolor="black")
for i, v in enumerate(counts.values):
    ax.text(i, v + 300, str(v), ha="center", fontweight="bold")
ax.set_title("Dataset Class Distribution", fontweight="bold")
plt.tight_layout()
plt.show()
print(f'Imbalance ratio: {counts[0]/counts[1]:.1f}:1')

## Section 4: Model Architecture

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, DistilBertModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

MAX_LENGTH           = 256   # 95th pct of email body length < 200 tokens
BATCH_SIZE           = 32    # fits T4 16GB VRAM at max_length=256
EPOCHS               = 3     # standard BERT fine-tuning; more = overfitting risk
LR                   = 2e-5  # AdamW standard for BERT fine-tuning
WARMUP_STEPS         = 100
CONFIDENCE_THRESHOLD = 0.70  # below this -> SUSPICIOUS at inference

In [ ]:
class PhishingEmailDataset(Dataset):
    """
    Stores tokenized email inputs + header auth features + labels.
    header_features: 10 SPF/DKIM/DMARC binary signals per email.
    """
    def __init__(self, input_ids, attn_masks, header_feats, labels):
        self.input_ids    = torch.tensor(input_ids,    dtype=torch.long)
        self.attn_masks   = torch.tensor(attn_masks,   dtype=torch.long)
        self.header_feats = torch.tensor(header_feats, dtype=torch.float32)
        self.labels       = torch.tensor(labels,       dtype=torch.long)

    def __len__(self): return len(self.labels)

    def __getitem__(self, i):
        return {
            "input_ids":       self.input_ids[i],
            "attention_mask":  self.attn_masks[i],
            "header_features": self.header_feats[i],
            "labels":          self.labels[i],
        }

print("PhishingEmailDataset defined.")

In [ ]:
class PhishingClassifier(nn.Module):
    """
    DistilBERT + header feature fusion for phishing detection.
    CLS embedding (768d) + projected header signals (32d) -> classifier.
    """
    def __init__(self):
        super().__init__()
        self.distilbert  = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.header_proj = nn.Linear(10, 32)
        self.classifier  = nn.Sequential(
            nn.Linear(768 + 32, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2),  # 0=LEGITIMATE, 1=PHISHING
        )

    def forward(self, input_ids, attention_mask, header_features):
        cls = self.distilbert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        hdr = torch.relu(self.header_proj(header_features))
        return self.classifier(torch.cat([cls, hdr], dim=1))

print("PhishingClassifier defined.")

In [ ]:
print("Tokenizing emails (this may take 3-5 min for large datasets)...")
tokenizer   = AutoTokenizer.from_pretrained("distilbert-base-uncased")
body_texts   = df["body_text"].fillna("").tolist()
splits_list  = df["split"].tolist()
labels_list  = df["label"].tolist()
header_feats = df[HEADER_FEATURE_NAMES].values.tolist()

encodings = tokenizer(
    body_texts, max_length=MAX_LENGTH,
    truncation=True, padding="max_length", return_tensors=None,
)

train_idx = [i for i, s in enumerate(splits_list) if s == "train"]
val_idx   = [i for i, s in enumerate(splits_list) if s == "val"]

def make_ds(idx):
    return PhishingEmailDataset(
        [encodings["input_ids"][i]      for i in idx],
        [encodings["attention_mask"][i]  for i in idx],
        [header_feats[i]                 for i in idx],
        [labels_list[i]                  for i in idx],
    )

train_ds = make_ds(train_idx)
val_ds   = make_ds(val_idx)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
train_labels = [labels_list[i] for i in train_idx]
print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,}")

## Section 5: Training

Fine-tuning DistilBERT for 3 epochs with weighted cross-entropy loss to handle class imbalance.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from transformers import get_linear_schedule_with_warmup

class_weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_labels)
weight_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion     = nn.CrossEntropyLoss(weight=weight_tensor)

model     = PhishingClassifier().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps
)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters : {n_params:,}")
print(f"Class weights    : Legit={class_weights[0]:.3f} | Phishing={class_weights[1]:.3f}")
print(f"Total train steps: {total_steps:,}")

In [ ]:
def evaluate(model, loader):
    """Evaluate model, return accuracy/precision/recall/f1/fpr/pr_auc."""
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score,
        f1_score, average_precision_score
    )
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for b in loader:
            logits = model(
                b["input_ids"].to(device),
                b["attention_mask"].to(device),
                b["header_features"].to(device),
            )
            all_probs.extend(torch.softmax(logits, 1)[:, 1].cpu().numpy())
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(b["labels"].numpy())
    preds  = np.array(all_preds)
    labels = np.array(all_labels)
    probs  = np.array(all_probs)
    fp = np.sum((preds == 1) & (labels == 0))
    tn = np.sum((preds == 0) & (labels == 0))
    return {
        "accuracy":  round(float(accuracy_score(labels, preds)), 4),
        "precision": round(float(precision_score(labels, preds, zero_division=0)), 4),
        "recall":    round(float(recall_score(labels, preds, zero_division=0)), 4),
        "f1":        round(float(f1_score(labels, preds, zero_division=0)), 4),
        "fpr":       round(float(fp / (fp + tn + 1e-9)), 4),
        "pr_auc":    round(float(average_precision_score(labels, probs)), 4),
    }

print("evaluate() function defined.")

In [ ]:
from tqdm.notebook import tqdm

best_f1   = 0.0
best_path = MODELS_DIR / "distilbert_tier1.pt"

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, n_batches = 0.0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", ncols=90)
    for b in pbar:
        ids  = b["input_ids"].to(device)
        mask = b["attention_mask"].to(device)
        hdrs = b["header_features"].to(device)
        lbls = b["labels"].to(device)
        optimizer.zero_grad()
        loss = criterion(model(ids, mask, hdrs), lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # prevent exploding gradients
        optimizer.step()
        scheduler.step()
        total_loss  += loss.item()
        n_batches   += 1
        pbar.set_postfix({"loss": f"{total_loss/n_batches:.4f}"})

    m = evaluate(model, val_loader)
    print(f"Epoch {epoch} | Loss={total_loss/n_batches:.4f} | "
          f"R={m['recall']:.4f} F1={m['f1']:.4f} FPR={m['fpr']:.4f} PR-AUC={m['pr_auc']:.4f}")

    if m["f1"] > best_f1:
        best_f1 = m["f1"]
        torch.save(model.state_dict(), best_path)
        print(f"  >> Best model checkpoint saved (F1={best_f1:.4f})")

print(f"\nTraining complete. Best val F1 = {best_f1:.4f}")

## Section 6: Results & Evaluation Plots

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=device))
final = evaluate(model, val_loader)

print("=" * 50)
print("Final Validation Metrics (DistilBERT Tier-1)")
print("=" * 50)
for k, v in final.items():
    print(f"  {k:12}: {v:.4f}")
print()
if final["recall"] >= 0.90: print("  Phishing Recall >= 0.90: ACHIEVED")
else:                        print(f"  Phishing Recall {final['recall']:.4f} < 0.90: not met")
if final["fpr"]    <= 0.05: print("  FPR <= 0.05: ACHIEVED")
else:                        print(f"  FPR {final['fpr']:.4f} > 0.05: not met")

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix as sk_cm, precision_recall_curve, average_precision_score, classification_report

model.eval()
p2, l2, pr2 = [], [], []
with torch.no_grad():
    for b in val_loader:
        logits = model(b["input_ids"].to(device), b["attention_mask"].to(device), b["header_features"].to(device))
        pr2.extend(torch.softmax(logits, 1)[:, 1].cpu().numpy())
        p2.extend(logits.argmax(1).cpu().numpy())
        l2.extend(b["labels"].numpy())
l2 = np.array(l2); p2 = np.array(p2); pr2 = np.array(pr2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("NETRA DistilBERT — Evaluation (Val Set)", fontsize=14, fontweight="bold")

cm = sk_cm(l2, p2, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["LEGITIMATE", "PHISHING"], yticklabels=["LEGITIMATE", "PHISHING"])
axes[0].set_title("Confusion Matrix"); axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

prec, rec, _ = precision_recall_curve(l2, pr2)
auc_val = average_precision_score(l2, pr2)
axes[1].plot(rec, prec, color="#e74c3c", lw=2, label=f"PR-AUC={auc_val:.4f}")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve"); axes[1].legend()

plt.tight_layout()
plt.savefig(EVAL_DIR / "distilbert_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()

print(classification_report(l2, p2, target_names=["LEGITIMATE", "PHISHING"]))

## Section 7: Save & Download Artifacts

Save the trained model, tokenizer, and inference config to Google Drive, then download to your local machine.

In [ ]:
# Save tokenizer
tokenizer_dir = MODELS_DIR / "distilbert_tokenizer"
tokenizer.save_pretrained(str(tokenizer_dir))
print("Tokenizer saved:", tokenizer_dir)

# Save inference config
db_config = {
    "model":                "distilbert-base-uncased",
    "max_length":           MAX_LENGTH,
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "header_features":      HEADER_FEATURE_NAMES,
    "best_val_f1":          final["f1"],
    "best_val_recall":      final["recall"],
    "best_val_fpr":         final["fpr"],
}
with open(MODELS_DIR / "distilbert_config.json", "w") as f:
    json.dump(db_config, f, indent=2)
print("Config saved: distilbert_config.json")

print("=" * 55)
print("All artifacts saved to Google Drive.")
print("Download these 3 items to your local ml/models/ folder:")
print("  1. distilbert_tier1.pt         (~250MB model weights)")
print("  2. distilbert_tokenizer/        (folder with all files)")
print("  3. distilbert_config.json       (inference configuration)")
print("=" * 55)

In [ ]:
# Download directly to browser
import zipfile
from google.colab import files

zip_path = "/content/distilbert_tokenizer.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    for fp in tokenizer_dir.rglob("*"):
        if fp.is_file():
            z.write(fp, fp.relative_to(tokenizer_dir.parent))

files.download(str(MODELS_DIR / "distilbert_tier1.pt"))
files.download(str(MODELS_DIR / "distilbert_config.json"))
files.download(zip_path)
print("Downloads initiated!")